# Desestructuración del PDF de CV Académico

Este cuaderno demuestra el problema de extracción de texto estándar en PDFs con múltiples columnas (como el CV académico institucional) y cómo la **extracción sensible al diseño (layout-aware)** resuelve el problema de la mezcla de columnas, permitiendo generar salidas estructuradas y compatibles con TypeScript.

## 1. El Problema: Extracción Horizontal Estándar

Cuando usamos `page.extract_text()`, pdfplumber junta el texto horizontalmente sin importar las columnas. Veamos cómo se ve la sección de tesis en el texto estándar:

In [24]:

import pdfplumber

pdf_path = "CV2026_WalterMata.pdf"

with pdfplumber.open(pdf_path) as pdf:
    # Buscamos la página donde empieza "Tesis o proyectos" (Página 12)
    page = pdf.pages[0]
    text = page.extract_text()
    
    print("--- TEXTO EXTRAÍDO HORIZONTALMENTE --- ")
    print(text)
    

--- TEXTO EXTRAÍDO HORIZONTALMENTE --- 
Sistema Institucional de Curriculum Vitae
Curriculum vitae
Walter Alexander Mata López
No. de trabajador: 3731
PERSONALES PRIVADOS
Correo institucional Lengua materna Teléfono
wmata@ucol.mx Español 3121073799
Correo adicional
waltermata@gmail.com
Domicilio particular
Colima, Mpio. Colima, Colima
DATOS LABORALES
ACADÉMICO
Nombramiento actual Fecha de ingreso a la institución
Profesor investigador de tiempo completo 1997-08-16
DES Unidad académica
Facultad de Ingeniería Mecánica y Eléctrica Facultad de IngenierÃa MecÃ¡nica y ElÃ©ctrica
Cuerpo académico Grado de consolidación
UCOL-CA-91 - AUTOMATIZACIÓN Y SISTEMAS
CAEC
EMBEBIDOS
Domicilio laboral
Km. 9 Carretera Colima - Coquimatlán NA Coquimatlán, Colima, 28400
Evaluar en: I.II.2
FORMACIÓN ACADÉMICA
Estudios realizados
........................................................................................................................................................................
Licenciatura 

Notarás que en la salida horizontal, el texto de los nombres de los estudiantes (columna izquierda) y los títulos de las tesis (columna derecha) se entrelazan porque se leen en la misma línea.

## 2. La Solución: Extracción Layout-Aware (Sensible al Diseño)

Para solucionar esto de manera robusta, podemos extraer las palabras individuales con sus coordenadas `x0` e `y` (representada por `top` en pdfplumber). 
Agrupamos las palabras por su línea vertical (`top` con una tolerancia de 3px) y luego las separamos en **Columna Izquierda** (`x0 < 280`) y **Columna Derecha** (`x0 >= 280`), insertando un tabulador (`\t`):

In [11]:
def extract_layout_aware_lines(page):
    words = page.extract_words()
    
    # Agrupar palabras por su coordenada vertical 'top'
    lines_dict = {}
    for w in words:
        top = w['top']
        found = False
        for t in lines_dict:
            if abs(t - top) < 3.0:
                lines_dict[t].append(w)
                found = True
                break
        if not found:
            lines_dict[top] = [w]
            
    # Procesar líneas ordenadas verticalmente
    lines = []
    for t in sorted(lines_dict.keys()):
        line_words = sorted(lines_dict[t], key=lambda w: w['x0'])
        
        left_col = []
        right_col = []
        for w in line_words:
            if w['x0'] < 280:
                left_col.append(w['text'])
            else:
                right_col.append(w['text'])
                
        left_str = " ".join(left_col).strip()
        right_str = " ".join(right_col).strip()
        
        # Siempre mantenemos la estructura de dos columnas separadas por tabulador
        lines.append(f"{left_str}\t{right_str}")
    return lines

with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[11]
    layout_lines = extract_layout_aware_lines(page)
    
    print("--- TEXTO EXTRAÍDO CON DETECCIÓN DE DISEÑO ---")
    for line in layout_lines[:25]:
        print(line)

--- TEXTO EXTRAÍDO CON DETECCIÓN DE DISEÑO ---
	Sistema Institucional de Curriculum Vitae
Ingeniería en Computación	
Licenciatura	No
Inteligente	
Estatus Organismo	acreditador Carga horaria semanal
	5
Semestre y	
No. de alumnos Periodo escolar	Año
grupo(s)	
2B 28 Feb-Jul	2025
Evaluar en: II	
........................................................................................................................................................................	
Curso impartido DES	Facultad
Facultad de	Ingeniería Mecánica Facultad de IngenierÃa
Programación para IA y CD	
y Eléctrica	MecÃ¡nica y ElÃ©ctrica
Programa educativo Nivel del programa	Acreditado
Maestría en Ingeniería Aplicada Maestría	Si
Estatus Organismo	acreditador Carga horaria semanal
Acreditado PNPC	6
Semestre y	
No. de alumnos Periodo escolar	Año
grupo(s)	
1ro, 1gpo 2 Ago-Ene	2025
Evaluar en: I.I.7	


## 3. Desestructurando las Columnas para la Salida Compatible con TS

Ahora que tenemos las columnas separadas por `\t`, podemos agrupar las líneas de un bloque de tesis y separar completamente la parte izquierda y la parte derecha:

In [12]:
import re

# Simulamos las líneas de un bloque de tesis del texto layout-aware
block_lines = [
    "Nombre del estudiante\tTítulo de la tesis",
    "Ana Isabel Sánchez Heredia, Rafael Zamora\tI-learning (Intelligent Learning) Aprendizaje",
    "Guerrero y Jesús Emilio Manzano Morales\tAsistido por Inteligencia Artificial",
    "Plantel\tNivel académico",
    "FACULTAD DE INGENIERIA MECANICA Y\t",
    "\tLicenciatura",
    "ELECTRICA\t",
    "Programa educativo\tTipo de dirección",
    "Ingeniería en Computación Inteligente\tAsesor",
    "Fecha de término\tFecha del examen",
    "2025-02-01\t"
]

# Separamos las columnas reuniendo todo el contenido izquierdo y derecho
left_parts = []
right_parts = []

for line in block_lines:
    parts = line.split("\t")
    if len(parts) >= 2:
        left_parts.append(parts[0].strip())
        right_parts.append(parts[1].strip())
    else:
        left_parts.append(parts[0].strip())

left_text = "\n".join([p for p in left_parts if p])
right_text = "\n".join([p for p in right_parts if p])

print("=== COLUMNA IZQUIERDA (DATOS ALUMNO) ===")
print(left_text)
print("\n=== COLUMNA DERECHA (DATOS TESIS) ===")
print(right_text)

=== COLUMNA IZQUIERDA (DATOS ALUMNO) ===
Nombre del estudiante
Ana Isabel Sánchez Heredia, Rafael Zamora
Guerrero y Jesús Emilio Manzano Morales
Plantel
FACULTAD DE INGENIERIA MECANICA Y
ELECTRICA
Programa educativo
Ingeniería en Computación Inteligente
Fecha de término
2025-02-01

=== COLUMNA DERECHA (DATOS TESIS) ===
Título de la tesis
I-learning (Intelligent Learning) Aprendizaje
Asistido por Inteligencia Artificial
Nivel académico
Licenciatura
Tipo de dirección
Asesor
Fecha del examen


## 4. Extracción Final y Mapeo a Estructura TypeScript

Al separar las columnas, podemos usar expresiones regulares en los textos resultantes de forma independiente y segura. 

Vea cómo calza exactamente con la interfaz `ThesisDirection` en `ScientificWritting.ts` / `listed.ts`:

In [13]:
student_name = re.search(r"Nombre del estudiante\n(.*?)\n(?:Plantel|$)", left_text, re.DOTALL).group(1).replace("\n", " ").strip()
thesis_title = re.search(r"Título de la tesis\n(.*?)\n(?:Nivel académico|$)", right_text, re.DOTALL).group(1).replace("\n", " ").strip()
plantel = re.search(r"Plantel\n(.*?)\n(?:Programa educativo|$)", left_text, re.DOTALL).group(1).replace("\n", " ").strip()
programa = re.search(r"Programa educativo\n(.*?)\n(?:Fecha de término|$)", left_text, re.DOTALL).group(1).replace("\n", " ").strip()
tipo_dir = re.search(r"Tipo de dirección\n(.*?)\n(?:Fecha del examen|$)", right_text, re.DOTALL).group(1).replace("\n", " ").strip()
fecha_termino = re.search(r"Fecha de término\n(.*?)$", left_text, re.DOTALL).group(1).replace("\n", " ").strip()

thesis_direction_ts_compatible = {
    "id": "thesis_1",
    "studentName": student_name,
    "thesisTitle": thesis_title,
    "institution": plantel,
    "program": programa,
    "directionType": tipo_dir,
    "endDate": fecha_termino
 }

import json
print(json.dumps(thesis_direction_ts_compatible, indent=2, ensure_ascii=False))

{
  "id": "thesis_1",
  "studentName": "Ana Isabel Sánchez Heredia, Rafael Zamora Guerrero y Jesús Emilio Manzano Morales",
  "thesisTitle": "I-learning (Intelligent Learning) Aprendizaje Asistido por Inteligencia Artificial",
  "institution": "FACULTAD DE INGENIERIA MECANICA Y ELECTRICA",
  "program": "Ingeniería en Computación Inteligente",
  "directionType": "Asesor",
  "endDate": "2025-02-01"
}
